# Traffic sign tracking, reading and playable video

This notebook controls the **same local frontend/backend** as SignTrace, using your existing **`gtsrb_best_model.pth`**. It does not retrain or replace the checkpoint.

The backend runs in the prepared **Python 3.12 `.venv312`** environment. The notebook client uses Python's standard library, so it does not depend on your old Python 3.15 kernel having PyTorch installed.

**Fixed:** browser-incompatible MPEG-4 output → H.264/yuv420p with fast start; missing dependencies → prepared backend environment; weak illustrated-sign predictions → explicit uncertainty; unread speed numbers → digit OCR; repeated alerts → spatial tracks and distinct observation votes. All source video frames are processed.

Keep this notebook, `traffic_sign_app/`, `.venv312/`, the checkpoint and video in the same project folder. Your original notebook/video/log are preserved in `traffic_sign_app/notebook_backup/`.

## 1. Paths and settings

Run from `/Users/neev/Downloads/DL`, or change `PROJECT_ROOT`. The default `real_road.mp4` sample is real street-driving footage; source attribution is in `traffic_sign_app/REAL_VIDEO_SOURCE.md`. The earlier illustrated `road.mp4` remains available. GTSRB contains photographed cropped signs, so full-scene localization is a separate heuristic step. Text readings are explicitly marked as OCR, and uncertain pictograms are not promoted to confident named detections.

In [ ]:
from pathlib import Path
import csv
import hashlib
import http.client
import json
import shutil
import subprocess
import time
import urllib.error
import urllib.request
import uuid
from IPython.display import display, Video, HTML

PROJECT_ROOT = Path('/Users/neev/Downloads/DL')
MODEL_PATH = PROJECT_ROOT / 'gtsrb_best_model.pth'
VIDEO_PATH = PROJECT_ROOT / 'real_road.mp4'
OUTPUT_VIDEO_PATH = PROJECT_ROOT / 'annotated_road.mp4'
ALERT_LOG_PATH = PROJECT_ROOT / 'traffic_sign_alerts.csv'
HOST, PORT = '127.0.0.1', 8000
SERVER_URL = f'http://{HOST}:{PORT}'
CONFIDENCE = 0.55
FRAME_STRIDE = 3
ENABLE_OCR = True
assert MODEL_PATH.is_file(), f'Missing checkpoint: {MODEL_PATH}'
assert VIDEO_PATH.is_file(), f'Missing video: {VIDEO_PATH}'

## 2. Start or reuse the local backend

The backend also serves the frontend at **http://127.0.0.1:8000**. If it is already running, this cell reuses it. If dependencies are missing, follow `traffic_sign_app/README.md`. FFmpeg is required for playable exports; Tesseract supplies text reading.

In [ ]:
def api_get(path):
    with urllib.request.urlopen(SERVER_URL + path, timeout=10) as response:
        return json.load(response)

def api_post(path, data=b'', content_type='application/x-www-form-urlencoded'):
    request = urllib.request.Request(SERVER_URL + path, data=data, method='POST',
                                     headers={'Content-Type': content_type})
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        raise RuntimeError(error.read().decode()) from error

try:
    health = api_get('/api/health')
except (OSError, urllib.error.URLError):
    executable = PROJECT_ROOT / '.venv312/bin/python'
    if not executable.is_file():
        raise RuntimeError('Create .venv312 and install traffic_sign_app/requirements.txt first.')
    log_path = PROJECT_ROOT / 'traffic_sign_app/runtime/notebook_server.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('ab') as log:
        SERVER_PROCESS = subprocess.Popen(
            [str(executable), '-m', 'uvicorn', 'traffic_sign_app.backend.main:app',
             '--host', HOST, '--port', str(PORT)],
            cwd=str(PROJECT_ROOT), stdout=log, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 40
    while True:
        try:
            health = api_get('/api/health')
            break
        except (OSError, urllib.error.URLError):
            if SERVER_PROCESS.poll() is not None or time.monotonic() > deadline:
                raise RuntimeError('Backend startup failed. Read: ' + str(log_path))
            time.sleep(0.5)
if not health['ready']:
    raise RuntimeError(health.get('error') or 'Model is unavailable.')
if not health['ffmpeg_available']:
    raise RuntimeError('Install FFmpeg: brew install ffmpeg')
print('Backend is ready:', SERVER_URL)
display(HTML(f'<a href="{SERVER_URL}" target="_blank">Open the interactive SignTrace frontend</a>'))

## 3. Verify the actual checkpoint

The reported validation score is stored in your training checkpoint; it is **not video detection accuracy**. The backend loads weights strictly and runs a forward-pass check. The SHA-256 check below confirms that the service uses your exact file.

In [ ]:
model_info = api_get('/api/model')
expected_sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
assert model_info['sha256'] == expected_sha256, 'The server is using a different model file.'
print('Architecture:', model_info['architecture'])
print('Classes:', model_info['classes'])
print('Inference device:', model_info['device'])
print('Stored best validation accuracy:', model_info['validation_accuracy'])
print('Text reader:', 'ready' if model_info['ocr_available'] else 'not installed')
print('Checkpoint SHA-256:', expected_sha256)

## 4. Submit the video

This streams your video to the local backend in chunks. It works with a different local video too: edit `VIDEO_PATH` above. The maximum file size is 250 MB. Classification uses your `.pth`; OCR reads speed numbers and sign-board text. Bounding boxes are matched one-to-one across frames, and cached OCR cannot count as repeated independent confirmation.

In [ ]:
def upload_local_video(path):
    path = Path(path)
    if path.stat().st_size > 250 * 1024 * 1024:
        raise ValueError('Video must not exceed 250 MB.')
    boundary = 'SignTrace' + uuid.uuid4().hex
    fields = {'threshold': str(CONFIDENCE), 'stride': str(FRAME_STRIDE),
              'ocr': str(ENABLE_OCR).lower()}
    prefix = b''
    for key, value in fields.items():
        prefix += (f'--{boundary}\r\nContent-Disposition: form-data; name="{key}"\r\n\r\n'
                   f'{value}\r\n').encode()
    safe_name = path.name.replace('"', '_').replace('\r', '_').replace('\n', '_')
    prefix += (f'--{boundary}\r\nContent-Disposition: form-data; name="file"; '
               f'filename="{safe_name}"\r\nContent-Type: video/mp4\r\n\r\n').encode()
    suffix = f'\r\n--{boundary}--\r\n'.encode()
    connection = http.client.HTTPConnection(HOST, PORT, timeout=60)
    try:
        connection.putrequest('POST', '/api/jobs')
        connection.putheader('Content-Type', f'multipart/form-data; boundary={boundary}')
        connection.putheader('Content-Length', len(prefix) + path.stat().st_size + len(suffix))
        connection.endheaders()
        connection.send(prefix)
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                connection.send(chunk)
        connection.send(suffix)
        response = connection.getresponse()
        result = json.loads(response.read())
        if response.status != 202:
            raise RuntimeError(result.get('detail', result))
        return result
    finally:
        connection.close()

job = upload_local_video(VIDEO_PATH)
JOB_ID = job['id']
print('Analysis started:', JOB_ID)


## 5. Wait for inference and H.264 export

This runs without `cv2.imshow()` or a separate OpenCV GUI window. Interrupting the cell requests cancellation on the backend. The web frontend can also show progress and cancel a job.

In [ ]:
last_progress = -1
try:
    while True:
        result = api_get('/api/jobs/' + JOB_ID)
        progress = int(result.get('progress', 0))
        if progress // 10 != last_progress // 10:
            print(f"{result['status']}: {progress}% — frame {result.get('frame', 0)}/{result.get('total_frames', '?')}")
            last_progress = progress
        if result['status'] == 'done':
            break
        if result['status'] in ['error', 'cancelled']:
            raise RuntimeError(result.get('error') or 'Analysis cancelled.')
        time.sleep(0.7)
except KeyboardInterrupt:
    api_post('/api/jobs/' + JOB_ID + '/cancel')
    print('Cancellation requested.')
    raise
print('Complete:', result['frame'], 'frames;', result['duration_seconds'], 'seconds.')

## 6. Save and play the corrected video

The output uses **H.264 + yuv420p + fast start**, which works in normal browser/Jupyter players. Original audio is preserved when available. The original broken output is backed up before replacement.

Voice alerts are synchronized to playback in the **web frontend**. The embedded notebook video contains source audio; it does not contain synthesized alert speech.

In [ ]:
def download_output(route, destination):
    destination = Path(destination)
    backup_dir = PROJECT_ROOT / 'traffic_sign_app/notebook_backup'
    backup_dir.mkdir(exist_ok=True)
    if destination.exists() and not (backup_dir / destination.name).exists():
        shutil.copy2(destination, backup_dir / destination.name)
    temporary = destination.with_name(destination.stem + '.partial' + destination.suffix)
    with urllib.request.urlopen(SERVER_URL + route, timeout=60) as source, temporary.open('wb') as target:
        shutil.copyfileobj(source, target)
    temporary.replace(destination)

download_output(result['video_url'], OUTPUT_VIDEO_PATH)
download_output(result['csv_url'], ALERT_LOG_PATH)
print('Video:', OUTPUT_VIDEO_PATH)
print('Readings:', ALERT_LOG_PATH)
display(Video(filename=str(OUTPUT_VIDEO_PATH), embed=True, html_attributes='controls playsinline'))

## 7. Review real readings and uncertainties

OCR confidence measures text recognition; GTSRB probabilities measure cropped-image classification. They are different evidence sources. A warning symbol that the checkpoint cannot identify reliably stays uncertain. No recognition results are fabricated from video timestamps.

In [ ]:
for event in result['events']:
    marker = 'POSSIBLE' if event['uncertain'] else 'READING'
    print(f"{event['time_seconds']:5.2f}s | {marker:8} | {event['label']} | "
          f"{event['source']} {event['confidence']:.0%}")
print('Confirmed readings:', result['confirmed_signs'])
print('Possible signs:', result['possible_signs'])

### Model and detection limits

Your saved MobileNetV3 checkpoint reports about **82.65% validation accuracy** on cropped GTSRB signs. It does not include a trained full-frame object detector. This application uses color/geometry proposals, spatial tracking and explicit text reading; it may miss unusual lighting, tiny signs, monochrome signs or signs outside the GTSRB classes. Text boards are read as text rather than being assigned invented GTSRB classes. It is a demonstration pipeline; inspect possible readings.

For broader real-world coverage, a separately trained traffic-sign object detector and representative video data are the next improvements. No retraining is performed here.